In [1]:
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install flask flask-cors pyngrok ollama

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 45 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (546 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently i

In [2]:
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
!ollama pull mistral
print("Mistral ready!")


Mistral ready!


In [3]:
from flask import Flask, jsonify, request
from flask_cors import CORS
import ollama, json, threading

app = Flask(__name__)
CORS(app)

sap_data = {"errors": [
    {"id":"ERR001","timestamp":"2025-03-24 08:12:33","source":"SAP ERP","module":"FI","transaction":"FB01","error_code":"POSTING_FAILED","severity":"CRITICAL","description":"Document posting failed in company code 1000. GL account 400000 is blocked for posting.","user":"JSMITH","system":"PRD"},
    {"id":"ERR002","timestamp":"2025-03-24 08:15:10","source":"SAP ERP","module":"MM","transaction":"MIGO","error_code":"STOCK_NEGATIVE","severity":"HIGH","description":"Goods receipt failed. Posting would lead to negative stock for material M-100 in plant 1000.","user":"AGARWAL","system":"PRD"},
    {"id":"ERR003","timestamp":"2025-03-24 08:22:45","source":"SAP MDG","module":"MDG-M","transaction":"MDG_BS_UI","error_code":"DUPLICATE_MATERIAL","severity":"HIGH","description":"Material master creation blocked. Duplicate material number detected: MAT-20045 already exists.","user":"KPATEL","system":"MDG"},
    {"id":"ERR004","timestamp":"2025-03-24 08:31:05","source":"SAP ERP","module":"SD","transaction":"VA01","error_code":"CREDIT_LIMIT_EXCEEDED","severity":"HIGH","description":"Sales order creation blocked. Customer C-5021 has exceeded credit limit by 15,000 USD.","user":"RLOPEZ","system":"PRD"},
    {"id":"ERR005","timestamp":"2025-03-24 08:45:22","source":"SAP ERP","module":"BASIS","transaction":"SM21","error_code":"DB_CONNECTION_TIMEOUT","severity":"CRITICAL","description":"Database connection timeout after 30 seconds. ABAP runtime error DBIF_REPO_INST_SQL_ERROR occurred.","user":"SYSTEM","system":"PRD"},
    {"id":"ERR006","timestamp":"2025-03-24 09:02:11","source":"SAP MDG","module":"MDG-F","transaction":"MDG_BS_UI","error_code":"WORKFLOW_STUCK","severity":"MEDIUM","description":"MDG workflow for vendor master change request CR-3345 pending approval for 48 hours.","user":"SYSTEM","system":"MDG"},
    {"id":"ERR007","timestamp":"2025-03-24 09:10:58","source":"SAP ERP","module":"PP","transaction":"CO01","error_code":"BOM_NOT_FOUND","severity":"MEDIUM","description":"Production order creation failed. Bill of materials for material FG-8801 not found in plant 2000.","user":"TCHEN","system":"PRD"},
    {"id":"ERR008","timestamp":"2025-03-24 09:25:30","source":"SAP ERP","module":"FI","transaction":"F110","error_code":"PAYMENT_RUN_FAILED","severity":"CRITICAL","description":"Automatic payment run F110 failed for company code 1000. Bank configuration missing for house bank HDFC01.","user":"SYSTEM","system":"PRD"},
    {"id":"ERR009","timestamp":"2025-03-24 09:40:15","source":"SAP MDG","module":"MDG-C","transaction":"BP","error_code":"INVALID_TAX_NUMBER","severity":"MEDIUM","description":"Customer master data rejected. Tax registration number format invalid for country IN.","user":"NRAO","system":"MDG"},
    {"id":"ERR010","timestamp":"2025-03-24 09:55:42","source":"SAP ERP","module":"BASIS","transaction":"SM37","error_code":"BATCH_JOB_CANCELLED","severity":"HIGH","description":"Background job SAP_REORG_JOBS cancelled after 2 hours. Insufficient memory in work process WP04.","user":"SYSTEM","system":"PRD"}
]}

def analyze_single_error(error):
    error_code = error.get('error_code') or error.get('code', 'UNKNOWN')
    prompt = f"""
You are a senior SAP consultant. Analyze this error and respond ONLY in this JSON format with no extra text:
{{
  "root_cause": "1-2 sentence explanation",
  "business_impact": "what process is affected and how badly",
  "recommended_fix": "step by step fix for IT team",
  "ai_priority": "CRITICAL or HIGH or MEDIUM or LOW"
}}

Error: {error.get('module','N/A')} | {error.get('transaction','N/A')} | {error_code} | {error.get('severity','N/A')}
Description: {error.get('message') or error.get('description','No description')}
"""
    response = ollama.chat(model='mistral', messages=[{'role': 'user', 'content': prompt}])
    try:
        text = response['message']['content']
        # Try direct parse first
        cleaned = text[text.find('{'):text.rfind('}')+1]
        parsed = json.loads(cleaned)
        # Validate all expected keys exist
        if all(k in parsed for k in ['root_cause', 'business_impact', 'recommended_fix', 'ai_priority']):
            return parsed
        raise ValueError("Missing keys")
    except:
        # Mistral returned plain text instead of JSON — extract manually
        text = response['message']['content']
        lines = text.strip().split('\n')
        result = {
            "root_cause":       "See full analysis below",
            "business_impact":  "See full analysis below",
            "recommended_fix":  "See full analysis below",
            "ai_priority":      error.get('severity', 'MEDIUM')
        }
        for line in lines:
            low = line.lower()
            if 'root cause' in low or 'root_cause' in low:
                result['root_cause'] = line.split(':', 1)[-1].strip().strip('"').strip(',')
            elif 'business impact' in low or 'business_impact' in low:
                result['business_impact'] = line.split(':', 1)[-1].strip().strip('"').strip(',')
            elif 'recommended' in low or 'fix' in low:
                result['recommended_fix'] = line.split(':', 1)[-1].strip().strip('"').strip(',')
        if result['root_cause'] == "See full analysis below":
            result['root_cause'] = text[:300]
        return result
def correlate_errors(errors):
    if len(errors) < 2:
        return []

    error_list = "\n".join([
        f"- [{e.get('id','?')}] {e.get('module','?')} | {e.get('code') or e.get('error_code','?')} | {e.get('severity','?')} | {e.get('message') or e.get('description','?')}"
        for e in errors
    ])

    prompt = f"""
You are a senior SAP consultant analyzing a set of SAP system errors detected together.

Here are all the flagged errors from the log scan:
{error_list}

Your job is to identify groups of RELATED errors that share a common root cause or are causing each other.

Respond ONLY in this exact JSON format with no extra text:
{{
  "correlation_groups": [
    {{
      "group_title": "short name for this group of related errors",
      "related_error_ids": ["ERR001", "ERR002"],
      "common_root_cause": "explanation of what is connecting these errors",
      "combined_business_impact": "what this group of errors means for the business together",
      "recommended_resolution_order": "which error to fix first and why"
    }}
  ],
  "standalone_errors": ["ERR003"],
  "overall_system_health": "CRITICAL or DEGRADED or WARNING or STABLE",
  "executive_summary": "2-3 sentence summary of the overall situation for management"
}}

Only group errors that are genuinely related. Errors with no connection go in standalone_errors.
"""

    response = ollama.chat(
        model='mistral',
        messages=[{'role': 'user', 'content': prompt}]
    )

    try:
        text = response['message']['content']
        return json.loads(text[text.find('{'):text.rfind('}')+1])
    except:
        return {
            "correlation_groups": [],
            "standalone_errors": [e.get('id','?') for e in errors],
            "overall_system_health": "UNKNOWN",
            "executive_summary": "Correlation analysis could not be parsed. Review individual errors above."
        }
@app.route('/api/health',  methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'message': 'SAP Monitor API is running'})

@app.route('/api/summary', methods=['GET'])
def summary():
    errors = sap_data['errors']
    return jsonify({'total': len(errors), 'CRITICAL': len([e for e in errors if e['severity']=='CRITICAL']), 'HIGH': len([e for e in errors if e['severity']=='HIGH']), 'MEDIUM': len([e for e in errors if e['severity']=='MEDIUM']), 'LOW': len([e for e in errors if e['severity']=='LOW'])})

@app.route('/api/errors',  methods=['GET'])
def get_errors():
    return jsonify(sap_data['errors'])

@app.route('/api/analyze/<error_id>', methods=['GET'])
def analyze_one(error_id):
    error = next((e for e in sap_data['errors'] if e['id'] == error_id), None)
    if not error: return jsonify({'error': 'Not found'}), 404
    return jsonify({**error, 'analysis': analyze_single_error(error)})

@app.route('/api/upload', methods=['POST'])
def upload_log():
    if 'file' not in request.files:
        return jsonify({'error': 'No file uploaded'}), 400
    file = request.files['file']
    if not file.filename.endswith('.json'):
        return jsonify({'error': 'JSON files only'}), 400
    try:
        content = json.loads(file.read().decode('utf-8'))

        if isinstance(content, list):
            raw_entries = content
        elif 'logs' in content:
            raw_entries = content['logs']
        elif 'errors' in content:
            raw_entries = content['errors']
        else:
            return jsonify({'error': 'Unrecognized log format'}), 400

        total_scanned = len(raw_entries)

        actionable = []
        for entry in raw_entries:
            log_type = entry.get('type', entry.get('severity', 'INFO')).upper()
            if log_type in ['ERROR', 'CRITICAL']:
                actionable.append({**entry, 'severity': log_type})
            elif log_type == 'WARNING':
                actionable.append({**entry, 'severity': 'MEDIUM'})

        if not actionable:
            return jsonify({
                'message': f'Scanned {total_scanned} log entries. No actionable issues found.',
                'summary': {'total': 0, 'total_scanned': total_scanned, 'CRITICAL': 0, 'HIGH': 0, 'MEDIUM': 0, 'LOW': 0},
                'results': []
            })

        results = []
        for entry in actionable:
            analysis = analyze_single_error(entry)
            results.append({**entry, 'analysis': analysis})

        order = {'CRITICAL': 0, 'ERROR': 1, 'HIGH': 1, 'MEDIUM': 2, 'WARNING': 2, 'LOW': 3}
        results.sort(key=lambda x: order.get(x.get('severity', 'LOW'), 3))

        summary = {
            'total':         len(results),
            'total_scanned': total_scanned,
            'CRITICAL':      len([e for e in results if e['severity'] == 'CRITICAL']),
            'HIGH':          len([e for e in results if e['severity'] in ['ERROR', 'HIGH']]),
            'MEDIUM':        len([e for e in results if e['severity'] in ['WARNING', 'MEDIUM']]),
            'LOW':           len([e for e in results if e['severity'] == 'LOW']),
        }

        # Run correlation analysis across all flagged errors
        print(f"Running correlation analysis on {len(results)} errors...")
        correlation = correlate_errors(actionable)

        return jsonify({
            'message': f'Scanned {total_scanned} log entries. Found {len(results)} issues requiring attention.',
            'summary': summary,
            'results': results,
            'correlation': correlation
        })

    except Exception as ex:
        return jsonify({'error': str(ex)}), 500

print("All endpoints ready!")

All endpoints ready!


In [4]:
from pyngrok import ngrok
import threading, time

ngrok.set_auth_token("2uwQnbQqG4mXm9ZGwrLJjHu1dQo_7WFc6dCzZ8Ew2Xd4znLpr")

def run_flask():
    app.run(port=5000, use_reloader=False, debug=False)

thread = threading.Thread(target=run_flask)
thread.daemon = True
thread.start()
time.sleep(2)

public_url = ngrok.connect(5000)
url = str(public_url).split('"')[1]
print("=" * 50)
print("YOUR API URL:", url)
print("=" * 50)
print("Update API_URL in your React App.js with this URL!")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


YOUR API URL: https://ea6b-34-125-79-168.ngrok-free.app
Update API_URL in your React App.js with this URL!
